In [1]:
import torch
import torchaudio
import librosa
import numpy as np
import parselmouth
from transformers import Wav2Vec2Model, Wav2Vec2Processor

SR = 16000
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load pretrained wav2vec2
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model.eval().to(device)

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=SR, mono=True)
    y = y / max(abs(y))
    
    # ---------------- Speaker features ----------------
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    
    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_mean = np.mean(mfccs.T, axis=0)
    
    energy = np.mean(y**2)
    
    speaker_feats = np.hstack([mfcc_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy])
    
    # ---------------- Question features ----------------
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec>0 else 0
    
    # Wav2Vec2 embeddings
    input_values = processor(y, sampling_rate=SR, return_tensors="pt", padding=True).input_values.to(device)
    with torch.no_grad():
        embeddings = wav2vec_model(input_values).last_hidden_state
        embeddings = embeddings.mean(dim=1).cpu().numpy().squeeze()
    
    question_feats = np.hstack([embeddings, speaking_rate])
    
    return speaker_feats, question_feats


C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 2571.61it/s]
Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch

root_folder = r'User_Voice'

X_speaker, X_question, y_user, y_question = [], [], [], []

for user_folder in sorted(os.listdir(root_folder)):
    user_path = os.path.join(root_folder, user_folder)
    if not os.path.isdir(user_path):
        continue
    for file in sorted(os.listdir(user_path)):
        if file.endswith('.wav'):
            file_path = os.path.join(user_path, file)
            sp_feats, q_feats = extract_features(file_path)
            X_speaker.append(sp_feats)
            X_question.append(q_feats)
            
            y_user.append(user_folder)
            q_num = int(file.split('.')[0])  # 01.wav -> 1
            y_question.append(q_num)

X_speaker = np.array(X_speaker)
X_question = np.array(X_question)
y_user = np.array(y_user)
y_question = np.array(y_question)

# Encode labels
le_user = LabelEncoder()
y_user_enc = le_user.fit_transform(y_user)
le_q = LabelEncoder()
y_question_enc = le_q.fit_transform(y_question)

print("Speaker features shape:", X_speaker.shape)
print("Question features shape:", X_question.shape)


Speaker features shape: (230, 45)
Question features shape: (230, 769)


In [3]:
class VoiceDataset(Dataset):
    def __init__(self, X_speaker, X_question, y_user, y_question):
        self.X_speaker = torch.tensor(X_speaker, dtype=torch.float32)
        self.X_question = torch.tensor(X_question, dtype=torch.float32)
        self.y_user = torch.tensor(y_user, dtype=torch.long)
        self.y_question = torch.tensor(y_question, dtype=torch.long)
    
    def __len__(self):
        return len(self.X_speaker)
    
    def __getitem__(self, idx):
        return self.X_speaker[idx], self.X_question[idx], self.y_user[idx], self.y_question[idx]

dataset = VoiceDataset(X_speaker, X_question, y_user_enc, y_question_enc)

train_size = int(0.8*len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [4]:
import torch.nn as nn

class MultiTaskNet(nn.Module):
    def __init__(self, sp_input_dim, q_input_dim, num_users, num_questions):
        super().__init__()
        # Speaker branch
        self.speaker_branch = nn.Sequential(
            nn.Linear(sp_input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        # Question branch
        self.question_branch = nn.Sequential(
            nn.Linear(q_input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        # Output heads
        self.user_head = nn.Linear(64, num_users)
        self.q_head = nn.Linear(128, num_questions)
    
    def forward(self, sp_x, q_x):
        sp_feat = self.speaker_branch(sp_x)
        q_feat = self.question_branch(q_x)
        user_out = self.user_head(sp_feat)
        q_out = self.q_head(q_feat)
        return user_out, q_out

model = MultiTaskNet(X_speaker.shape[1], X_question.shape[1], len(le_user.classes_), len(le_q.classes_))
model.to(device)


MultiTaskNet(
  (speaker_branch): Sequential(
    (0): Linear(in_features=45, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
  )
  (question_branch): Sequential(
    (0): Linear(in_features=769, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
  )
  (user_head): Linear(in_features=64, out_features=23, bias=True)
  (q_head): Linear(in_features=128, out_features=10, bias=True)
)

In [5]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for sp_x, q_x, y_u, y_q in train_loader:
        sp_x, q_x, y_u, y_q = sp_x.to(device), q_x.to(device), y_u.to(device), y_q.to(device)
        optimizer.zero_grad()
        out_u, out_q = model(sp_x, q_x)
        loss_u = criterion(out_u, y_u)
        loss_q = criterion(out_q, y_q)
        loss = loss_u + loss_q
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1, Loss: 7.2806
Epoch 2, Loss: 4.6229
Epoch 3, Loss: 3.9995
Epoch 4, Loss: 3.4557
Epoch 5, Loss: 3.0065
Epoch 6, Loss: 2.5113
Epoch 7, Loss: 2.0908
Epoch 8, Loss: 1.7571
Epoch 9, Loss: 1.5492
Epoch 10, Loss: 1.3654
Epoch 11, Loss: 1.2279
Epoch 12, Loss: 1.1073
Epoch 13, Loss: 1.0321
Epoch 14, Loss: 0.8994
Epoch 15, Loss: 0.8320
Epoch 16, Loss: 0.8036
Epoch 17, Loss: 0.7374
Epoch 18, Loss: 0.6644
Epoch 19, Loss: 0.6560
Epoch 20, Loss: 0.6553
Epoch 21, Loss: 0.6287
Epoch 22, Loss: 0.6119
Epoch 23, Loss: 0.5356
Epoch 24, Loss: 0.5180
Epoch 25, Loss: 0.4906
Epoch 26, Loss: 0.4825
Epoch 27, Loss: 0.4476
Epoch 28, Loss: 0.4527
Epoch 29, Loss: 0.4267
Epoch 30, Loss: 0.4105


In [6]:
from sklearn.metrics import accuracy_score

model.eval()
all_user_preds, all_user_labels = [], []
all_q_preds, all_q_labels = [], []

with torch.no_grad():
    for sp_x, q_x, y_u, y_q in test_loader:
        sp_x, q_x = sp_x.to(device), q_x.to(device)
        out_u, out_q = model(sp_x, q_x)
        user_pred = out_u.argmax(dim=1).cpu().numpy()
        q_pred = out_q.argmax(dim=1).cpu().numpy()
        all_user_preds.extend(user_pred)
        all_user_labels.extend(y_u.numpy())
        all_q_preds.extend(q_pred)
        all_q_labels.extend(y_q.numpy())

print("Speaker accuracy:", accuracy_score(all_user_labels, all_user_preds))
print("Question accuracy:", accuracy_score(all_q_labels, all_q_preds))


Speaker accuracy: 1.0
Question accuracy: 0.5434782608695652


In [9]:
def predict(file_path, model, le_user, le_q, device='cpu'):
    model.eval()
    
    # Extract features
    sp_feats, q_feats = extract_features(file_path)
    
    # Convert to tensor
    sp_tensor = torch.tensor(sp_feats, dtype=torch.float32).unsqueeze(0).to(device)
    q_tensor = torch.tensor(q_feats, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out_user, out_q = model(sp_tensor, q_tensor)
        user_pred = le_user.inverse_transform([out_user.argmax(dim=1).cpu().item()])[0]
        q_pred = le_q.inverse_transform([out_q.argmax(dim=1).cpu().item()])[0]
    
    return user_pred, q_pred

# Example usage:
file = "User_Voice/U023/09.wav"
user_pred, q_pred = predict(file, model, le_user, le_q, device)
print(f"Predicted User: {user_pred}, Predicted Question: {q_pred}")


Predicted User: U023, Predicted Question: 9


In [10]:
# Speaker model
model_speaker = nn.Sequential(
    nn.Linear(X_speaker.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, len(le_user.classes_))
)

# Question model
model_question = nn.Sequential(
    nn.Linear(X_question.shape[1], 256),
    nn.ReLU(),
    nn.Linear(256, len(le_q.classes_))
)
